# Anexo III · Laboratório: o rabo, o cachorro e a volatilidade
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**: VIX e S&P 500 desde 1990, 20 ações da B3 desde 2005 (com a limpeza do Capítulo 28) e o CDI diário do Banco Central (série SGS 12). Na primeira execução, o notebook baixa os dados do Yahoo Finance e do Banco Central na sua máquina e guarda uma cópia local (`dados_anexo3_opcoes.pkl`), que as execuções seguintes reaproveitam. O Yahoo corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. As opções das Partes 4 e 5 são teóricas (Black e Scholes com a volatilidade passada; Parte 5 com delta, gama, theta e vega). Testes didáticos, sem custo: **não são recomendação**.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import re
import time
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from scipy.stats import norm

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
TICKERS = [t + ".SA" for t in B3]
FIM = "2026-10-01"
CACHE = "dados_anexo3_opcoes.pkl"
PRAZO, JANELA_VOL = 42, 63                 # opção de dois meses (42 pregões); volatilidade dos 63 pregões anteriores


def mostra(*partes):
    """print com vírgula decimal e ponto de milhar (padrão brasileiro)."""
    texto = " ".join(str(p) for p in partes)
    texto = re.sub(r"(?<=\d)\.(?=\d)", ",", texto)
    print(re.sub(r"(?<=\d)_(?=\d)", ".", texto))


def baixa(*args, **kw):
    for tentativa in range(5):                 # o provedor às vezes falha: tenta de novo
        try:
            x = yf.download(*args, progress=False, **kw)
            if len(x):
                return x
        except Exception:
            pass
        time.sleep(5 * (tentativa + 1))
    ativo = args[0] if isinstance(args[0], str) else f"a lista de {len(args[0])} ativos"
    raise RuntimeError(f"Não consegui baixar {ativo} do Yahoo Finance depois de 5 tentativas. O provedor pode estar"
                       " fora do ar ou limitando pedidos. Espere alguns minutos e rode a célula de novo.")


def cdi_bcb(ini=2005, fim=2026):
    """CDI diário (% ao dia) do Banco Central, série SGS 12, em blocos de até 10 anos."""
    partes = []
    for a in range(ini, fim + 1, 10):
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados?formato=json"
               f"&dataInicial=01/01/{a}&dataFinal=31/12/{min(a + 9, fim)}")
        for tentativa in range(5):
            try:
                js = requests.get(url, timeout=60).json()
                break
            except Exception:
                time.sleep(5 * (tentativa + 1))
        else:
            raise RuntimeError("Não consegui baixar o CDI do Banco Central depois de 5 tentativas. O site pode estar"
                               " fora do ar. Espere alguns minutos e rode a célula de novo.")
        partes.append(pd.Series({pd.to_datetime(x["data"], dayfirst=True): float(x["valor"]) / 100 for x in js}))
    return pd.concat(partes).sort_index()

## Parte 1: o VIX contra a volatilidade que veio depois

In [ ]:
if os.path.exists(CACHE):                  # cópia local: o provedor pode corrigir o histórico entre execuções
    vix, spx, dados, cdi = pd.read_pickle(CACHE)
else:
    vix = baixa("^VIX", start="1990-01-01", end=FIM, auto_adjust=False, multi_level_index=False)["Close"]
    spx = baixa("^GSPC", start="1990-01-01", end=FIM, auto_adjust=False, multi_level_index=False)["Close"]
    dados = baixa(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, group_by="ticker")
    cdi = cdi_bcb()
    pd.to_pickle((vix, spx, dados, cdi), CACHE)
pc = lambda x, k=1: f"{x:.{k}%}"

r = np.log(spx).diff()
real_fut = r[::-1].rolling(21).std()[::-1].shift(-1) * np.sqrt(252) * 100   # realizada nos 21 pregões seguintes
df = pd.DataFrame({"vix": vix, "realizada": real_fut}).dropna()
df["prêmio"] = df["vix"] - df["realizada"]
mostra(f"Parte 1. VIX contra a volatilidade realizada nos 21 pregões seguintes ({df.index[0]:%Y} a {df.index[-1]:%Y},"
       f" {len(df):_} dias)")
mostra(f"   VIX acima da realizada em {pc((df['prêmio'] > 0).mean(), 0)} dos dias | diferença mediana"
       f" {df['prêmio'].median():.1f} pontos | VIX médio {df['vix'].mean():.1f} | realizada média {df['realizada'].mean():.1f}")
for a, b in ((0, 15), (15, 20), (20, 30), (30, 200)):
    g = df[(df["vix"] >= a) & (df["vix"] < b)]
    mostra(f"   VIX de {a} a {b if b < 200 else 'mais'}: {len(g):>5_} dias | VIX acima da realizada em"
           f" {pc((g['prêmio'] > 0).mean(), 0)} | diferença mediana {g['prêmio'].median():.1f}")

In [ ]:
import matplotlib.pyplot as plt
m = df.loc["2005":]
plt.figure(figsize=(10, 3.5))
plt.semilogy(m["vix"], color="black", lw=0.7, label="VIX")
plt.semilogy(m["realizada"], color="gray", lw=0.7, label="realizada nos 21 pregões seguintes")
plt.legend()
plt.show()

## Parte 2: as maiores surpresas

In [ ]:
mostra("\nParte 2. As cinco maiores surpresas (realizada menos VIX, em pontos de volatilidade)")
sur = (df["realizada"] - df["vix"]).sort_values(ascending=False)
vistos = []
for dt, v in sur.items():
    if all(abs((dt - x).days) > 60 for x in vistos):
        vistos.append(dt)
        mostra(f"   {dt:%d/%m/%Y}: VIX {df.at[dt, 'vix']:.1f} | realizada depois {df.at[dt, 'realizada']:.1f}"
               f" | surpresa {v:.1f}")
    if len(vistos) == 5:
        break
SURPRESAS = vistos

## Parte 3: o VIX como termômetro

In [ ]:
mostra("\nParte 3. Retorno do S&P 500 nos 21 pregões seguintes, por faixa do VIX")
ret21 = spx.shift(-21) / spx - 1
tab = pd.DataFrame({"vix": vix, "ret": ret21}).dropna()
tab["faixa"] = pd.cut(tab["vix"], [0, 12, 15, 20, 25, 30, 40, 200],
                      labels=["< 12", "12-15", "15-20", "20-25", "25-30", "30-40", "40+"])
mostra("   faixa      dias   média  mediana  positivos  5% piores")
for faixa, g in tab.groupby("faixa", observed=True)["ret"]:
    mostra(f"   {faixa:6s} {len(g):>8_} {pc(g.mean()):>7s} {pc(g.median()):>8s} {pc((g > 0).mean(), 0):>10s}"
           f" {pc(g.quantile(0.05)):>10s}")

CORRIGIDAS = {}                            # o que a limpeza mudou (registro da Parte 7; sem repetir)
SUSPEITOS = {"EQTL3.SA": ("2010-08-25", "queda de 40% sem volta (provável ajuste de bonificação ou desdobramento faltando)")}


def limpa(d, nome=""):
    """Limpeza do Capítulo 26, mais três regras para dado sujo que engana o teste de stops:
    1) máxima impossível: pavio de mais de 50% acima da abertura e do fechamento vira o maior dos dois;
    2) desdobramento ou grupamento sem ajuste: salto de mais de 2 vezes (ou menos da metade) da abertura em relação ao
       fechamento anterior, sem volta nos 5 pregões seguintes, corrige o histórico anterior pela mesma proporção;
    3) buraco nos dados: com mais de 15 dias corridos sem cotação, só fica o trecho contínuo depois do último buraco."""
    d = d.dropna().copy()
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] > d["Low"])
    d = d[ok & (d["Volume"] > 0)] if "Volume" in d else d[ok]
    px = ["Open", "High", "Low", "Close"]
    topo = d[["Open", "Close"]].max(axis=1)
    alta = d["High"] > 1.5 * topo
    for dt in d.index[alta]:
        CORRIGIDAS.setdefault(f"{nome} {dt:%d/%m/%Y}: máxima {d.at[dt, 'High']:.2f} com fechamento {d.at[dt, 'Close']:.2f}", None)
    d.loc[alta, "High"] = topo[alta]
    salto = d["Open"] / d["Close"].shift()
    dura = d["Close"].shift(-5) / d["Close"].shift()
    for dt in d.index[((salto < 0.5) | (salto > 2)) & ((dura < 0.6) | (dura > 1.6))]:
        f_ = salto.loc[dt]
        d.loc[d.index < dt, px] *= f_
        CORRIGIDAS.setdefault(f"{nome} {dt:%d/%m/%Y}: salto de {f_:.4f} vezes sem volta (ajuste que faltou); histórico"
                          " anterior corrigido", None)
    if nome == "SUZB3.SA":                 # antes disso, preço parado no Yahoo (Capítulo 26)
        d = d.loc["2017-11-10":]
    if nome in SUSPEITOS:                  # queda sem volta que parece ajuste faltando, mas sem prova: fica fora
        dt, motivo = SUSPEITOS[nome]
        CORRIGIDAS.setdefault(f"{nome} {pd.Timestamp(dt):%d/%m/%Y}: {motivo}; usa só a partir dessa data", None)
        d = d.loc[dt:]
    buraco = d.index.to_series().diff().dt.days
    if (buraco > 15).any():
        ini = buraco[buraco > 15].index[-1]
        CORRIGIDAS.setdefault(f"{nome}: {int((buraco > 15).sum())} buracos de mais de 15 dias; usa só a partir de {ini:%d/%m/%Y}", None)
        d = d.loc[ini:]
    return d



def bs_call(S, K, T, r_, sig):
    """Prêmio de uma opção de compra europeia (Black e Scholes, 1973), com juro contínuo r_."""
    d1 = (np.log(S / K) + (r_ + sig ** 2 / 2) * T) / (sig * np.sqrt(T))
    return S * norm.cdf(d1) - K * np.exp(-r_ * T) * norm.cdf(d1 - sig * np.sqrt(T))


def gregas(S, K, T, r_, sig):
    d1 = (np.log(S / K) + (r_ + sig ** 2 / 2) * T) / (sig * np.sqrt(T))
    d2 = d1 - sig * np.sqrt(T)
    delta = norm.cdf(d1)
    theta = -S * norm.pdf(d1) * sig / (2 * np.sqrt(T)) - r_ * K * np.exp(-r_ * T) * norm.cdf(d2)   # por ano
    vega = S * norm.pdf(d1) * np.sqrt(T)                                                             # por 1,00 de vol
    gama = norm.pdf(d1) / (S * sig * np.sqrt(T))
    return delta, theta, vega, gama

## Parte 4: subir não basta

In [ ]:
juro = np.log(1 + cdi) * 252                                    # juro contínuo ao ano, a partir do CDI diário
linhas = []
for tk in TICKERS:
    d = limpa(dados[tk], tk)
    c = d["Close"]
    vol = np.log(c).diff().rolling(JANELA_VOL).std() * np.sqrt(252)
    rr = juro.reindex(c.index).ffill()
    for i in range(JANELA_VOL + 1, len(c) - PRAZO, PRAZO // 2):  # uma opção nova a cada 21 pregões
        S0, ST, sig, r_ = c.iat[i], c.iat[i + PRAZO], vol.iat[i], rr.iat[i]
        if not np.isfinite(sig) or not np.isfinite(r_) or sig <= 0:
            continue
        premio = bs_call(S0, S0, PRAZO / 252, r_, sig)
        linhas.append({"ação": tk, "data": c.index[i], "subiu": ST > S0, "alta": ST / S0 - 1,
                       "premio_pct": premio / S0, "resultado": max(ST - S0, 0) / premio - 1})
OP = pd.DataFrame(linhas)
sub = OP[OP["subiu"]]
mostra(f"\nParte 4. Opções de compra no dinheiro, de dois meses, em {len(TICKERS)} ações da B3: {len(OP):_} opções"
       f" ({OP['data'].min():%Y} a {OP['data'].max():%Y})")
mostra(f"   prêmio mediano: {pc(OP['premio_pct'].median())} do preço da ação")
mostra(f"   a ação subiu em {pc(OP['subiu'].mean(), 0)} das vezes; quem comprou a opção perdeu dinheiro em"
       f" {pc((OP['resultado'] < 0).mean(), 0)} das vezes, e perdeu tudo em {pc((OP['resultado'] <= -1).mean(), 0)}")
mostra(f"   quando a ação subiu, o comprador da opção ainda perdeu em {pc((sub['resultado'] < 0).mean(), 0)} das vezes"
       f" (a alta não pagou o prêmio)")
mostra(f"   resultado médio do comprador: {pc(OP['resultado'].mean(), 0)} do prêmio; mediano: {pc(OP['resultado'].median(), 0)}")
mostra("   (prêmio com a volatilidade passada; no mercado, a implícita costuma ser maior, e o prêmio, mais caro)")

## Parte 5: o gráfico da opção mistura três coisas

In [ ]:
d = limpa(dados["PETR4.SA"], "PETR4.SA")
c = d["Close"]
sextas = [dt for dt in c.index if dt.weekday() == 4 and 15 <= dt.day <= 21]   # terceira sexta-feira do mês
venc = max(dt for dt in sextas if dt <= c.index[-1])
fim_i = c.index.get_loc(venc)
ini_i = fim_i - PRAZO
vol21 = np.log(c).diff().rolling(21).std() * np.sqrt(252)       # a "implícita" do exemplo: volatilidade de 21 pregões
rr = juro.reindex(c.index).ffill()
K = round(c.iat[ini_i], 2)
cam = []
for i in range(ini_i, fim_i + 1):
    T = max((fim_i - i) / 252, 1e-6)
    S, sig, r_ = c.iat[i], vol21.iat[i], rr.iat[i]
    p = bs_call(S, K, T, r_, sig) if i < fim_i else max(S - K, 0)
    cam.append({"data": c.index[i], "S": S, "sig": sig, "T": T, "r": r_, "premio": p})
CAM = pd.DataFrame(cam).set_index("data")
contrib = {"delta": 0.0, "gama": 0.0, "theta": 0.0, "vega": 0.0, "resto": 0.0}
for a, b in zip(CAM.index[:-1], CAM.index[1:]):
    x, y = CAM.loc[a], CAM.loc[b]
    de, th, ve, ga = gregas(x["S"], K, x["T"], x["r"], x["sig"])
    p_delta, p_theta, p_vega = de * (y["S"] - x["S"]), th * (1 / 252), ve * (y["sig"] - x["sig"])
    p_gama = 0.5 * ga * (y["S"] - x["S"]) ** 2                   # curvatura: o delta muda quando a ação anda
    contrib["delta"] += p_delta
    contrib["gama"] += p_gama
    contrib["theta"] += p_theta
    contrib["vega"] += p_vega
    contrib["resto"] += (y["premio"] - x["premio"]) - p_delta - p_gama - p_theta - p_vega
mostra(f"\nParte 5. PETR4: opção de compra de dois meses, preço de exercício R$ {K:.2f}, de {CAM.index[0]:%d/%m/%Y}"
       f" a {venc:%d/%m/%Y} (vencimento)")
mostra(f"   ação: de R$ {CAM['S'].iat[0]:.2f} para R$ {CAM['S'].iat[-1]:.2f} ({pc(CAM['S'].iat[-1] / CAM['S'].iat[0] - 1)})"
       f" | prêmio: de R$ {CAM['premio'].iat[0]:.2f} para R$ {CAM['premio'].iat[-1]:.2f}"
       f" ({pc(CAM['premio'].iat[-1] / CAM['premio'].iat[0] - 1, 0)})")
mostra(f"   volatilidade usada: de {pc(CAM['sig'].iat[0], 0)} para {pc(CAM['sig'].iat[-2], 0)} na véspera do vencimento")
mostra("   de onde veio a variação do prêmio (R$ por ação): " + " | ".join(f"{k} {v:+.2f}" for k, v in contrib.items()))

In [ ]:
import matplotlib.pyplot as plt
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
a1.plot(CAM["S"], color="black"); a1.axhline(K, ls=":", color="gray"); a1.set_ylabel("PETR4")
a2.plot(CAM["premio"], color="black"); a2.set_ylabel("prêmio da opção")
plt.show()

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, as funções `bs_call` e `gregas`, na célula da Parte 3, e o laço da Parte 5 que separa a variação do prêmio. Mostre por que o theta é multiplicado por `1 / 252` e de onde vem a parcela `p_gama`." Confira com a Parte 5, que imprime a variação do prêmio da opção de PETR4 separada em delta (+6,39), gama (+0,85), theta (−1,33) e vega (+0,27).
2. **Adaptar:** "Copie a Parte 5 para uma célula nova e troque a primeira linha por `d = limpa(baixa('[ATIVO]', start='2005-01-01', end=FIM, auto_adjust=True, multi_level_index=False), '[ATIVO]')`, com o ticker do Yahoo Finance. Não mude `bs_call`, `gregas` nem a volatilidade de 21 pregões." Antes de olhar o ativo novo, confira se a Parte 5 original continua dando o prêmio de R$ 1,99 para R$ 8,11 e o theta de −1,33, como no livro.

Mais pedidos deste anexo (entender e aplicar ao seu gráfico) estão na página do anexo, no hub do livro.

## Para ir além
1. Na Parte 1, troque os 21 pregões por 63. O VIX continua acima da realizada em 85% dos dias?
2. Na Parte 4, some 20% à volatilidade usada no prêmio (a implícita costuma ser maior). Quantas vezes o comprador perde agora?
3. Na Parte 4, troque a opção no dinheiro por uma 5% fora do dinheiro (preço de exercício 5% acima).
4. Na Parte 5, escolha outra ação e outro vencimento e veja quanto o theta tirou do prêmio.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.